In [ ]:
# ============================================================
# 03_Linear_Regression.ipynb
# Section 1 — Setup, Paths, and Data Loading
# Purpose: Load preprocessed FD001 data, verify shapes,
#          prepare for Linear Regression modeling.
# ============================================================

# --- 1a. Mount Drive ---

# --- 1b. Canonical paths (same as all notebooks) ---
import os

PROJECT_ROOT = os.path.join(DATA_DIR, "..")
DATA_DIR      = os.path.join(PROJECT_ROOT, "data")
PROCESSED_DIR = os.path.join(DATA_DIR, "processed")
RESULTS_DIR   = os.path.join(PROJECT_ROOT, "results")
FIGURES_DIR   = os.path.join(PROJECT_ROOT, "figures")
EXPERIMENT_DIR = os.path.join(PROJECT_ROOT, "experiments")
SRC_DIR       = os.path.join(PROJECT_ROOT, "src")

import sys
sys.path.insert(0, SRC_DIR)

# --- 1c. Imports ---
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error
from preprocessing import FINAL_FEATURES_FD001

# --- 1d. Load preprocessed data ---
train = pd.read_parquet(os.path.join(PROCESSED_DIR, "train_FD001_processed.parquet"))
test_last = pd.read_parquet(os.path.join(PROCESSED_DIR, "test_FD001_last_cycle.parquet"))

print("=== Data loaded from 02_Preprocessing output ===\n")
print(f"Train shape:     {train.shape}")
print(f"Test-last shape: {test_last.shape}")
print(f"\nFeature columns ({len(FINAL_FEATURES_FD001)}):")
print(FINAL_FEATURES_FD001)
print(f"\nTrain RUL columns: {[c for c in train.columns if 'RUL' in c]}")
print(f"Test-last columns: {list(test_last.columns)}")

# --- 1e. Quick verification ---
assert train.shape[0] == 20631, f"Expected 20631 train rows, got {train.shape[0]}"
assert test_last.shape[0] == 100, f"Expected 100 test engines, got {test_last.shape[0]}"
assert all(f in train.columns for f in FINAL_FEATURES_FD001), "Missing features in train"
assert "RUL_linear" in train.columns, "Missing RUL_linear in train"
assert "RUL_piecewise125" in train.columns, "Missing RUL_piecewise125 in train"
assert "RUL_true" in test_last.columns, "Missing RUL_true in test_last"

print("\n✓ All checks passed. Ready for modeling.")

In [ ]:
# ============================================================
# 03_Linear_Regression.ipynb
# Section 2 — Define Evaluation Metrics
# Purpose: MAE, RMSE, NASA Score (asymmetric), and a helper
#          function that computes all three at once.
# Source: NASA Score from Saxena et al. 2008 (PHM08 Challenge).
# ============================================================

def nasa_score(y_true, y_pred):
    """
    PHM08 asymmetric scoring function (Saxena et al. 2008).

    d = predicted - true
    d < 0 (early, conservative): exp(-d/13) - 1
    d >= 0 (late, dangerous):    exp( d/10) - 1

    Lower is better. Penalizes late predictions more heavily.
    """
    d = y_pred - y_true
    scores = np.where(d < 0,
                      np.exp(-d / 13.0) - 1.0,
                      np.exp( d / 10.0) - 1.0)
    return float(np.sum(scores))


def evaluate_model(y_true, y_pred, model_name="Model", label_name="label"):
    """
    Compute MAE, RMSE, and NASA Score for a single model run.
    Prints a summary and returns a dict of results.
    """
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    ns   = nasa_score(y_true, y_pred)

    print(f"--- {model_name} | {label_name} ---")
    print(f"  MAE:        {mae:.2f} cycles")
    print(f"  RMSE:       {rmse:.2f} cycles")
    print(f"  NASA Score: {ns:.2f}")
    print()

    return {
        "model": model_name,
        "rul_label": label_name,
        "mae": round(mae, 2),
        "rmse": round(rmse, 2),
        "nasa_score": round(ns, 2),
    }


# --- Quick sanity check on the NASA Score function ---
# A perfect prediction (d=0 for all) should give score = 0
perfect = nasa_score(np.array([100, 50, 25]), np.array([100, 50, 25]))
assert perfect == 0.0, f"Perfect prediction should score 0, got {perfect}"

# An early prediction (d=-10) should be penalized less than a late one (d=+10)
early_penalty = nasa_score(np.array([100]), np.array([90]))   # d = -10
late_penalty  = nasa_score(np.array([100]), np.array([110]))  # d = +10
assert late_penalty > early_penalty, "Late penalty should exceed early penalty"

print("=== Evaluation metrics defined ===\n")
print(f"Sanity check — perfect prediction score: {perfect}")
print(f"Sanity check — early (d=-10) penalty:    {early_penalty:.4f}")
print(f"Sanity check — late  (d=+10) penalty:    {late_penalty:.4f}")
print(f"Sanity check — late > early:             {late_penalty > early_penalty}")
print("\n✓ NASA Score function verified. Ready to evaluate models.")

In [ ]:
# ============================================================
# 03_Linear_Regression.ipynb
# Section 3 — Train and Evaluate Linear Regression
# Purpose: Train LR on both RUL labels, predict on test-last,
#          compute MAE, RMSE, NASA Score, and training time.
# ============================================================

# --- 3a. Prepare feature matrices ---
X_train = train[FINAL_FEATURES_FD001].values
X_test  = test_last[FINAL_FEATURES_FD001].values
y_true  = test_last["RUL_true"].values

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape:  {X_test.shape}")
print(f"y_true shape:  {y_true.shape}")
print(f"y_true range:  {y_true.min()} to {y_true.max()}\n")

# ============================================================
# 3b. Run 1 — Linear Regression with LINEAR RUL label
# ============================================================
y_train_linear = train["RUL_linear"].values

start_time = time.time()
lr_linear = LinearRegression()
lr_linear.fit(X_train, y_train_linear)
train_time_linear = time.time() - start_time

y_pred_linear = lr_linear.predict(X_test)

result_linear = evaluate_model(y_true, y_pred_linear,
                               model_name="Linear Regression",
                               label_name="linear")
result_linear["training_time_sec"] = round(train_time_linear, 4)

# ============================================================
# 3c. Run 2 — Linear Regression with PIECEWISE RUL label
# ============================================================
y_train_pw = train["RUL_piecewise125"].values

start_time = time.time()
lr_piecewise = LinearRegression()
lr_piecewise.fit(X_train, y_train_pw)
train_time_pw = time.time() - start_time

y_pred_piecewise = lr_piecewise.predict(X_test)

result_piecewise = evaluate_model(y_true, y_pred_piecewise,
                                  model_name="Linear Regression",
                                  label_name="piecewise125")
result_piecewise["training_time_sec"] = round(train_time_pw, 4)

# ============================================================
# 3d. Summary comparison
# ============================================================
print("=" * 60)
print("SUMMARY — Linear Regression Baseline")
print("=" * 60)
print(f"{'Metric':<16} {'Linear RUL':>12} {'Piecewise RUL':>14}")
print("-" * 60)
print(f"{'MAE':<16} {result_linear['mae']:>12.2f} {result_piecewise['mae']:>14.2f}")
print(f"{'RMSE':<16} {result_linear['rmse']:>12.2f} {result_piecewise['rmse']:>14.2f}")
print(f"{'NASA Score':<16} {result_linear['nasa_score']:>12.2f} {result_piecewise['nasa_score']:>14.2f}")
print(f"{'Train time (s)':<16} {result_linear['training_time_sec']:>12.4f} {result_piecewise['training_time_sec']:>14.4f}")
print("=" * 60)

In [ ]:
# ============================================================
# 03_Linear_Regression.ipynb
# Section 4 — Visualization: Predicted vs True RUL
# Purpose: Scatter plots (pred vs true) and per-engine error
#          plots for both labels. Saved as thesis figures.
# Paths: uses FIGURES_DIR from Section 1.
# ============================================================

# --- 4a. Predicted vs True scatter — both labels side by side ---
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax, y_pred, label_name, res in [
    (axes[0], y_pred_linear,    "Linear RUL label",      result_linear),
    (axes[1], y_pred_piecewise, "Piecewise-125 RUL label", result_piecewise),
]:
    ax.scatter(y_true, y_pred, alpha=0.6, s=35, color="#2E75B6",
               edgecolor="white", linewidth=0.5)

    # Perfect-prediction diagonal
    lims = [0, max(y_true.max(), y_pred.max()) * 1.05]
    ax.plot(lims, lims, "r--", linewidth=1.2, label="Perfect prediction")

    ax.set_xlabel("True RUL (cycles)")
    ax.set_ylabel("Predicted RUL (cycles)")
    ax.set_title(f"Linear Regression — {label_name}\n"
                 f"MAE {res['mae']:.2f} | RMSE {res['rmse']:.2f} | "
                 f"NASA {res['nasa_score']:.0f}")
    ax.legend(loc="upper left")
    ax.set_xlim(lims)
    ax.set_ylim(lims)

fig.suptitle("FD001 — Linear Regression: predicted vs. true RUL "
             "(100 test engines, last cycle)", fontsize=13, y=1.02)
fig.tight_layout()

fig_path = os.path.join(FIGURES_DIR, "fd001_lr_pred_vs_true.png")
fig.savefig(fig_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved figure to:\n{fig_path}\n")

# --- 4b. Error vs True RUL — where does the model fail? ---
fig2, axes2 = plt.subplots(1, 2, figsize=(14, 5))

for ax, y_pred, label_name in [
    (axes2[0], y_pred_linear,    "Linear RUL label"),
    (axes2[1], y_pred_piecewise, "Piecewise-125 RUL label"),
]:
    errors = y_pred - y_true   # positive = late (dangerous), negative = early (safe)
    ax.scatter(y_true, errors, alpha=0.6, s=35, color="#D9822B",
               edgecolor="white", linewidth=0.5)
    ax.axhline(0, color="black", linewidth=1)
    ax.set_xlabel("True RUL (cycles)")
    ax.set_ylabel("Prediction error (pred − true, cycles)")
    ax.set_title(f"{label_name}\nAbove 0 = late prediction (dangerous), "
                 "below 0 = early (conservative)")

fig2.suptitle("FD001 — Linear Regression: prediction error vs. true RUL",
              fontsize=13, y=1.03)
fig2.tight_layout()

fig2_path = os.path.join(FIGURES_DIR, "fd001_lr_error_vs_true.png")
fig2.savefig(fig2_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved figure to:\n{fig2_path}\n")

# --- 4c. Quantify errors by RUL zone (for the notes) ---
print("=== Error analysis by RUL zone (piecewise model) ===\n")
errors_pw = y_pred_piecewise - y_true

zones = [("Near failure (RUL <= 30)",  y_true <= 30),
         ("Mid life (30 < RUL <= 80)", (y_true > 30) & (y_true <= 80)),
         ("Early life (RUL > 80)",     y_true > 80)]

for zone_name, mask in zones:
    n = mask.sum()
    zone_mae = np.abs(errors_pw[mask]).mean()
    zone_bias = errors_pw[mask].mean()
    print(f"{zone_name:<28} n={n:>3} | MAE {zone_mae:>6.2f} | "
          f"mean error {zone_bias:>+7.2f} "
          f"({'late/over' if zone_bias > 0 else 'early/under'})")

In [ ]:
# ============================================================
# 03_Linear_Regression.ipynb
# Section 5 — Save Predictions and Finalize
# Purpose: Save per-engine predictions (both labels) to results/,
#          print final summary for the experiment log.
# Paths: uses RESULTS_DIR from Section 1.
# ============================================================

# --- 5a. Build the per-engine predictions table ---
predictions = pd.DataFrame({
    "unit": test_last["unit"].values,
    "cycle_last": test_last["cycle"].values,
    "RUL_true": y_true,
    "pred_LR_linear": np.round(y_pred_linear, 2),
    "pred_LR_piecewise125": np.round(y_pred_piecewise, 2),
})
predictions["error_linear"] = np.round(predictions["pred_LR_linear"] - predictions["RUL_true"], 2)
predictions["error_piecewise125"] = np.round(predictions["pred_LR_piecewise125"] - predictions["RUL_true"], 2)

print("Per-engine predictions table (first 10 rows):")
print(predictions.head(10).to_string(index=False))

# --- 5b. Save to results/ ---
pred_path = os.path.join(RESULTS_DIR, "fd001_lr_predictions.csv")
predictions.to_csv(pred_path, index=False)
print(f"\nSaved predictions to:\n{pred_path}")

# --- 5c. Final summary block (for experiment log + thesis notes) ---
print("\n" + "=" * 60)
print("NOTEBOOK 03 COMPLETE — Final Results Summary")
print("=" * 60)
print(f"""
Model: LinearRegression (scikit-learn, default settings)
Features: 13 sensors (final set from Notebook 02)
Test protocol: last available cycle per test engine (n=100)

Run 1 — linear label:
  MAE {result_linear['mae']}, RMSE {result_linear['rmse']}, NASA {result_linear['nasa_score']}, time {result_linear['training_time_sec']}s

Run 2 — piecewise125 label:
  MAE {result_piecewise['mae']}, RMSE {result_piecewise['rmse']}, NASA {result_piecewise['nasa_score']}, time {result_piecewise['training_time_sec']}s

Files saved:
  results/fd001_lr_predictions.csv
  figures/fd001_lr_pred_vs_true.png
  figures/fd001_lr_error_vs_true.png
""")
print("=" * 60)

# Notebook 03 — Linear Regression Baseline: Complete Documentation

## Purpose

Train and evaluate the first model of the complexity ladder: Linear
Regression, the simplest possible regression baseline. The model is
trained twice — once with the linear RUL label and once with the
piecewise-125 label — and evaluated on the 100 test engines at their
last available cycle.

**Position in the project.** First of four modeling notebooks
(03 Linear Regression → 04 XGBoost → 05 LSTM → 06 Chronos). All four use
the identical frozen data from `02_Preprocessing.ipynb` and the identical
evaluation protocol, so that differences in results are attributable to
the models alone.

**Why Linear Regression as baseline.** It is the simplest model that can
solve a regression task: one weight per feature plus a bias. It trains in
a fraction of a second and has no hyperparameters to tune. If a more
complex model cannot clearly beat it, the added complexity is not
justified — which is exactly the primary research question of this thesis.

## Inputs and outputs

**Inputs (from `data/processed/`):**
- `train_FD001_processed.parquet` (20631 rows, 13 features + 2 RUL labels)
- `test_FD001_last_cycle.parquet` (100 rows, 13 features + RUL_true)

**Outputs:**
- `results/fd001_lr_predictions.csv` — per-engine predictions, both labels
- `figures/fd001_lr_pred_vs_true.png` — predicted vs. true scatter
- `figures/fd001_lr_error_vs_true.png` — error structure diagnostic
- Experiment log rows EXP-001 (linear) and EXP-002 (piecewise125)

## Section overview

**Section 1 — Setup and data loading.** Loads the frozen preprocessed
data. No re-preprocessing happens in modeling notebooks — the pipeline
was locked in Notebook 02. Feature list imported from
`src/preprocessing.py` (13 sensors: s2, s3, s4, s7, s8, s9, s11, s12,
s13, s15, s17, s20, s21). Verified: train (20631, 17), test-last (100, 16).

**Section 2 — Evaluation metrics.** MAE and RMSE from scikit-learn; NASA
Score implemented from its published definition (Saxena et al. 2008,
PHM 2008 Challenge). The NASA Score is asymmetric: for error
d = predicted − true, early predictions (d < 0) score exp(−d/13) − 1,
late predictions (d ≥ 0) score exp(d/10) − 1, summed over all engines,
lower is better. Late predictions (claiming more remaining life than the
engine has) are penalized more heavily, because an unexpected in-service
failure is worse than an unnecessarily early maintenance action. The
constants 13 and 10 are fixed by the benchmark, not tunable. Sanity
checks passed: perfect prediction scores 0; late penalty > early penalty.

**Section 3 — Training and evaluation.** LinearRegression (scikit-learn,
default settings, deterministic, no seed needed). No tuning — consistent
with thesis scope (Section 1.4): models are configured to reasonable
documented settings, not maximally tuned. Test protocol: one prediction
per test engine at its last available cycle vs. RUL_FD001.txt ground
truth (n = 100), identical for all models.

**Section 4 — Visualization and error analysis.** Predicted-vs-true
scatter and error-vs-true plots for both labels; quantitative error
analysis by RUL zone.

**Section 5 — Save.** Per-engine prediction table (both labels, with
errors) saved to `results/fd001_lr_predictions.csv` for traceability and
for reuse by Notebook 07 (comparison) without retraining.

## Results (recorded, final run)

| Run | Label | MAE | RMSE | NASA Score | Train time (s) |
|---|---|---|---|---|---|
| EXP-001 | linear | 25.64 | 32.20 | 14093.94 | 0.0894 |
| EXP-002 | piecewise125 | 17.79 | 22.07 | 1334.57 | 0.0239 |

**Error structure by RUL zone (piecewise model):**

| Zone | n | MAE | Mean error |
|---|---|---|---|
| Near failure (RUL ≤ 30) | 25 | 19.85 | +18.77 (late/over — dangerous) |
| Mid life (30 < RUL ≤ 80) | 20 | 19.76 | +19.75 (late/over) |
| Early life (RUL > 80) | 55 | 16.14 | −8.46 (early/under — conservative) |

## Main findings

1. **The piecewise label improves every metric dramatically** — −31 % MAE,
   −31 % RMSE, −90 % NASA Score vs. the linear label — with the same
   model, same features, same data. The label definition alone is a major
   performance factor: first direct evidence for the secondary research
   question (labelling ablation).
2. **The NASA Score gap is driven by over-prediction.** The linear label
   permits large over-predictions during the healthy phase; the
   asymmetric score punishes these exponentially. The piecewise label
   implicitly caps predictions (visible in the scatter: predictions
   plateau near the 125 training cap).
3. **Error structure = regression to the mean.** The error cloud slopes
   downward with true RUL: nearly-failed engines are over-predicted
   (+18.77 mean bias — the operationally dangerous direction, dominant
   contributor to the NASA Score), healthy engines are under-predicted
   (−8.46 — merely conservative). The linear model pulls all predictions
   toward the training-distribution average and cannot commit to extremes.
4. **Training cost ≈ 0** (< 0.1 s) — the reference point of the
   cost-accuracy ladder.

## Open question for the next models

Do more complex models — especially sequence models that can read the
steep late-life sensor trends found in the EDA (Sections 7 and 12) —
reduce the dangerous near-failure over-prediction?

**Next:** `04_XGBoost.ipynb` — same data, same protocol, same metrics.
Key questions: accuracy gain from non-linearity, its training cost, and
whether the near-failure bias is reduced.

# Linear Regression — Strengths, Weaknesses, and Handoff to XGBoost

## Strengths (Vorteile)

1. **Essentially free to train.** 0.02–0.09 seconds. No hyperparameters,
   no tuning, no GPU. This is the anchor of the cost-accuracy ladder.

2. **Fully interpretable.** One weight per sensor — the contribution of
   each sensor to the prediction can be read directly. No black box.

3. **Deterministic and reproducible.** Same data → same model, always.
   No random seeds, no variance between runs.

4. **Already benefits enormously from the right label.** With the
   piecewise label it reaches MAE 17.79 — a respectable baseline.

5. **No overfitting risk in practice.** 13 features, 20631 training rows,
   only 14 parameters total (13 weights + 1 bias). The model is far too
   simple to memorize the data.

## Weaknesses (Nachteile) — from the recorded results

1. **Linearity assumption.** The model assumes each sensor contributes to
   RUL proportionally, everywhere. But the EDA (Sections 7 and 12) proved
   the degradation signal is exponential: sensors are almost flat in
   early life and change steeply in the last ~50 cycles. A straight line
   cannot represent "flat, then steep." The model is structurally
   mismatched to the documented physics.

2. **Regression to the mean → the dangerous near-failure bias.** From the
   zone analysis: nearly-failed engines (RUL ≤ 30) are over-predicted by
   +18.77 cycles on average — the model says "30 cycles left" when the
   engine has ~10. This is the operationally dangerous direction and the
   dominant contributor to the NASA Score. Early-life engines are
   under-predicted (−8.46) — merely conservative. Because the model
   cannot bend its prediction curve, it pulls everything toward the
   middle of the training distribution and cannot commit to extremes.

3. **No feature interactions.** Each sensor is treated independently: the
   effect of s11 on RUL is the same regardless of what s9 or s4 show.
   Physically, degradation appears as a combination of signals. Linear
   Regression has no way to represent "if A is high AND B is low, then…".

4. **Single-row view (no temporal context).** The model sees only the
   current cycle's values — not the trajectory. Engines degrade at rates
   varying by 3.3× (EDA Section 11), so two engines with identical
   current readings can have different RUL depending on how fast they are
   degrading. A single-row model cannot see rate.
   *(Note: this weakness is shared by XGBoost — it is addressed by the
   LSTM in Notebook 05, not by Notebook 04.)*

## The precise problem statement

> Linear Regression's core problem on FD001: it imposes a globally linear
> sensor→RUL relationship on data whose degradation signal is exponential
> and interactive. The consequence is regression to the mean — and its
> most dangerous expression is the systematic over-prediction of
> nearly-failed engines (+18.77 cycles mean bias in the RUL ≤ 30 zone),
> which is exactly the error mode the NASA Score penalizes hardest.

## Handoff to XGBoost (Notebook 04)

| Weakness | Addressed by XGBoost? | How |
|---|---|---|
| 1. Linearity | ✅ Yes — primary target | Trees split the feature space into regions: "if s11 > 0.8 → low-RUL region." Can represent flat-then-steep. |
| 2. Near-failure bias | ✅ Testable hypothesis | If linearity causes the bias, XGBoost should reduce the +18.77 mean error. The zone analysis in Notebook 04 measures this directly. |
| 3. No interactions | ✅ Yes | Tree paths are AND-conditions: "s11 high AND s4 high → predict 15 cycles." |
| 4. No temporal context | ❌ No — deliberately unsolved | XGBoost still sees single rows. Carried forward as the motivation for the LSTM (Notebook 05). |

**Cost side (primary research question):** these gains are not free.
XGBoost trains hundreds of trees — training time rises from ~0.02 s to
seconds. Small in absolute terms, but the ratio matters for the ladder.

## The ladder logic this creates

1. **LR → XGBoost:** fixes linearity + interactions.
   Test: does MAE drop, and does the near-failure bias shrink?
2. **XGBoost → LSTM:** fixes temporal blindness.
   Test: does reading trajectories (degradation rate) help further,
   given the 3.3× rate heterogeneity?
3. **LSTM → Chronos:** tests whether pretrained general time-series
   knowledge adds anything beyond a task-trained sequence model.

Each ladder step targets a named, measured weakness of the previous
model — a reasoned progression, not model-shopping.

In [ ]:
# ============================================================
# Figure 6.1 — Experimental pipeline flowchart
# ============================================================
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch
import os

FIGURES_DIR = os.path.join(DATA_DIR, "..", "figures")

fig, ax = plt.subplots(figsize=(12, 7))
ax.set_xlim(0, 12); ax.set_ylim(0, 10); ax.axis("off")

def box(x, y, w, h, text, fc="#E8EEF7", ec="#2E75B6"):
    ax.add_patch(FancyBboxPatch((x, y), w, h,
                 boxstyle="round,pad=0.08", fc=fc, ec=ec, linewidth=1.5))
    ax.text(x + w/2, y + h/2, text, ha="center", va="center",
            fontsize=9.5, wrap=True)

def arrow(x1, y1, x2, y2):
    ax.add_patch(FancyArrowPatch((x1, y1), (x2, y2),
                 arrowstyle="-|>", mutation_scale=16,
                 color="#555555", linewidth=1.3))

# Shared pipeline (top)
box(0.3, 8.3, 2.6, 1.2, "FD001 raw data\n(train + test)", fc="#F2F2F2", ec="#888888")
box(3.3, 8.3, 2.9, 1.2, "Preprocessing\n(feature removal,\nnormalization) — frozen", fc="#F2F2F2", ec="#888888")
box(6.6, 8.3, 2.6, 1.2, "Two RUL labels\n(linear /\npiecewise-125)", fc="#F2F2F2", ec="#888888")
box(9.6, 8.3, 2.1, 1.2, "Engine-level\nvalidation split\n(80 / 20)", fc="#F2F2F2", ec="#888888")
arrow(2.9, 8.9, 3.3, 8.9)
arrow(6.2, 8.9, 6.6, 8.9)
arrow(9.2, 8.9, 9.6, 8.9)

# Four model columns
models = [
    ("Linear\nRegression", "no tuning", 0.3),
    ("XGBoost", "6-config\nsweep", 3.3),
    ("LSTM", "6-config\nsweep", 6.6),
    ("Chronos-2\n+ head", "6-config\nsweep", 9.6),
]
colors = ["#999999", "#2E75B6", "#C00000", "#548235"]
for (name, tune, x), c in zip(models, colors):
    box(x, 5.6, 2.1, 1.3, name, fc="#FFFFFF", ec=c)
    box(x, 4.0, 2.1, 1.1, f"Tune on\nvalidation\n({tune})", fc="#FBFBFB", ec=c)
    box(x, 2.4, 2.1, 1.1, "Train on all\n100 engines", fc="#FBFBFB", ec=c)
    arrow(x + 1.05, 5.6, x + 1.05, 5.1)
    arrow(x + 1.05, 4.0, x + 1.05, 3.5)

# Funnel from validation split to the four models
for _, _, x in models:
    arrow(10.6, 8.3, x + 1.05, 6.9)

# Evaluation + comparison (bottom)
box(2.5, 0.6, 7.0, 1.2,
    "Evaluation on 100 test engines (last cycle):\nMAE · RMSE · NASA Score · training cost",
    fc="#E8EEF7", ec="#2E75B6")
for _, _, x in models:
    arrow(x + 1.05, 2.4, min(max(x + 1.05, 3.0), 9.0), 1.8)

ax.text(6, 9.75, "Shared, identical pipeline", ha="center",
        fontsize=10, style="italic", color="#555555")

fig.tight_layout()
out = os.path.join(FIGURES_DIR, "fig_6_1_pipeline.png")
fig.savefig(out, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {out}")